In [1]:
#第18章/加载数据集
from datasets import load_dataset
import torchvision
import torch


def get_dataset():
    #加载数据集
    dataset = load_dataset('lansinuote/gen.5.flower.book', split='train')

    #删除多余的字段
    dataset = dataset.remove_columns(['cls'])

    #图像数据预处理
    compose = torchvision.transforms.Compose([
        torchvision.transforms.Resize(224),
        torchvision.transforms.ToTensor(),
    ])

    def f(data):
        image = compose(data['image'][0]).unsqueeze(dim=0)
        return {'image': image}

    dataset = dataset.with_transform(f)

    #为了加速数据遍历的效率,把全体数据载入内存以加速IO
    dataset_tensor = torch.empty(len(dataset), 3, 224, 224)

    for i in range(len(dataset)):
        dataset_tensor[i] = dataset[i]['image']

    return dataset_tensor


dataset = get_dataset()

dataset.shape, dataset.dtype

Using custom data configuration lansinuote--gen.5.flower.book-34a531175c385260
Found cached dataset parquet (/root/.cache/huggingface/datasets/lansinuote___parquet/lansinuote--gen.5.flower.book-34a531175c385260/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


(torch.Size([2000, 3, 224, 224]), torch.float32)

In [2]:
#第18章/定义loader
loader = torch.utils.data.DataLoader(dataset=dataset,
                                     batch_size=4,
                                     shuffle=True,
                                     drop_last=True)

len(loader), next(iter(loader)).shape

(500, torch.Size([4, 3, 224, 224]))

In [3]:
#第18章/show函数
def show(images):
    from matplotlib import pyplot as plt

    images = images.to('cpu').detach()[:4]
    images = images.permute(0, 2, 3, 1)

    plt.figure(figsize=(20, 10))

    for i in range(len(images)):
        plt.subplot(1, 4, i + 1)
        plt.imshow(images[i])
        plt.axis('off')

    plt.show()


show(next(iter(loader)))

<Figure size 2000x1000 with 4 Axes>

In [4]:
#第18章/定义GEN模型
from transformers import PreTrainedModel, PretrainedConfig


class GEN(PreTrainedModel):
    config_class = PretrainedConfig

    def __init__(self, config):
        super().__init__(config)

        self.encoder = torch.nn.Sequential(
            torch.nn.ReflectionPad2d(padding=4),
            torch.nn.Conv2d(3, 32, kernel_size=9, stride=1, padding=0),
            torch.nn.InstanceNorm2d(32, affine=True),
            torch.nn.ReLU(),
            torch.nn.ReflectionPad2d(padding=1),
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=0),
            torch.nn.InstanceNorm2d(64, affine=True),
            torch.nn.ReLU(),
            torch.nn.ReflectionPad2d(padding=1),
            torch.nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=0),
            torch.nn.InstanceNorm2d(128, affine=True),
            torch.nn.ReLU(),
        )

        self.middle = torch.nn.ModuleList([
            torch.nn.Sequential(
                torch.nn.ReflectionPad2d(padding=1),
                torch.nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=0),
                torch.nn.InstanceNorm2d(128, affine=True),
                torch.nn.ReLU(),
                torch.nn.ReflectionPad2d(padding=1),
                torch.nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=0),
                torch.nn.InstanceNorm2d(128, affine=True),
            ) for _ in range(5)
        ])

        self.decoder = torch.nn.Sequential(
            torch.nn.UpsamplingNearest2d(scale_factor=2),
            torch.nn.ReflectionPad2d(padding=1),
            torch.nn.Conv2d(128, 64, kernel_size=3, stride=1, padding=0),
            torch.nn.InstanceNorm2d(64, affine=True),
            torch.nn.ReLU(),
            torch.nn.UpsamplingNearest2d(scale_factor=2),
            torch.nn.ReflectionPad2d(padding=1),
            torch.nn.Conv2d(64, 32, kernel_size=3, stride=1, padding=0),
            torch.nn.InstanceNorm2d(32, affine=True),
            torch.nn.ReLU(),
            torch.nn.ReflectionPad2d(padding=4),
            torch.nn.Conv2d(32, 3, kernel_size=9, stride=1, padding=0),
        )

    def forward(self, data):
        data = self.encoder(data)

        for i in self.middle:
            data = data + i(data)

        data = self.decoder(data)

        return data


gen = GEN(PretrainedConfig())

gen(torch.randn(2, 3, 224, 224)).shape

torch.Size([2, 3, 224, 224])

In [5]:
#第18章/加载预训练的CNN模型
import torchvision

cnn = torchvision.models.vgg16(weights='IMAGENET1K_V1').features[:19]
cnn.eval()

len(cnn)

19

In [6]:
#第18章/获取内容特征的函数
def get_feature_content(image):
    for i, layer in enumerate(cnn):
        image = layer(image)

        if i == 6:
            return image


get_feature_content(torch.randn(2, 3, 224, 224)).shape

torch.Size([2, 128, 112, 112])

In [7]:
#第18章/获取风格特征的函数
def get_feature_style(image):
    feature = []
    for i, layer in enumerate(cnn):
        image = layer(image)

        if i in [1, 6, 11, 18]:
            fenmu = image.shape[1] * image.shape[2] * image.shape[3]

            data = image.flatten(start_dim=2)
            data = torch.matmul(data, data.permute(0, 2, 1))

            data = data / fenmu

            feature.append(data)

    return feature


for i in get_feature_style(torch.randn(2, 3, 224, 224)):
    print(i.shape)

torch.Size([2, 64, 64])
torch.Size([2, 128, 128])
torch.Size([2, 256, 256])
torch.Size([2, 512, 512])


In [8]:
#第18章/加载style图片的特征图
#因为全局只需要计算一次，所以这里直接把结果算出来
def get_target_style():
    import PIL.Image

    #加载style图片
    image_style = PIL.Image.open('./datas/style.jpg')

    #处理style图片
    compose = torchvision.transforms.Compose([
        torchvision.transforms.Resize(250),
        torchvision.transforms.CenterCrop(224),
        torchvision.transforms.ToTensor(),
        lambda x: x.unsqueeze(dim=0),
    ])
    image_style = compose(image_style)

    show(image_style)

    #抽取style特征
    return get_feature_style(image_style)


target_style = get_target_style()

for i in target_style:
    print(i.shape)

<Figure size 2000x1000 with 1 Axes>

torch.Size([1, 64, 64])
torch.Size([1, 128, 128])
torch.Size([1, 256, 256])
torch.Size([1, 512, 512])


In [9]:
#第18章/训练前的准备工作
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(gen.parameters(), lr=2e-4)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

cnn.to(device)
gen.to(device)
target_style = [i.to(device).detach() for i in target_style]

cnn.eval()
gen.train()

device

'cuda'

In [10]:
#第18章/训练
def train():
    for epoch in range(20):

        for i, image in enumerate(loader):
            image = image.to(device)

            with torch.no_grad():
                target_content = get_feature_content(image)

            image_gen = gen(image)

            feature_content = get_feature_content(image_gen)
            feature_style = get_feature_style(image_gen)

            loss_style = sum(
                [criterion(i, j) for i, j in zip(feature_style, target_style)])
            loss_content = criterion(feature_content, target_content)

            loss = 1e5 * loss_content + 1e10 * loss_style
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        if epoch % 2 == 0:
            print(epoch, 1e5 * loss_content.item(), 1e10 * loss_style.item())
            show(image_gen.clip(0, 1))

    torch.save(gen.to('cpu'), 'save/gen.model')


train()

/root/anaconda3/envs/pt39/lib/python3.9/site-packages/torch/nn/modules/loss.py:530: UserWarning: Using a target size (torch.Size([1, 64, 64])) that is different to the input size (torch.Size([4, 64, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/root/anaconda3/envs/pt39/lib/python3.9/site-packages/torch/nn/modules/loss.py:530: UserWarning: Using a target size (torch.Size([1, 128, 128])) that is different to the input size (torch.Size([4, 128, 128])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/root/anaconda3/envs/pt39/lib/python3.9/site-packages/torch/nn/modules/loss.py:530: UserWarning: Using a target size (torch.Size([1, 256, 256])) that is different to the input size (torch.Size([4, 256, 256])). This will likely lead to incorrect results due

0 81229.3529510498 32879.70230303472


<Figure size 2000x1000 with 4 Axes>

2 54837.23282814026 28523.299988592044


<Figure size 2000x1000 with 4 Axes>

...

<Figure size 2000x1000 with 4 Axes>

16 38957.980275154114 18773.32238109375


<Figure size 2000x1000 with 4 Axes>

18 37668.24007034302 20115.530787734315


<Figure size 2000x1000 with 4 Axes>

In [11]:
#第18章/测试
gen = torch.load('save/gen.model')

with torch.no_grad():
    pred = gen(next(iter(loader))).clip(0, 1)

show(pred)

<Figure size 2000x1000 with 4 Axes>

In [12]:
#第18章/在线加载笔者训练好的模型并测试
gen = GEN.from_pretrained('lansinuote/gen.8.style_transfer.book')

with torch.no_grad():
    pred = gen(next(iter(loader))).clip(0, 1)

show(pred)

<Figure size 2000x1000 with 4 Axes>